# Загрузка и подготовка данных dataset_rework

**Шаг 1 плана.** Загрузка dataset_rework, подготовка сессий, переименование signal_barrier → rd_regime, добавление rd_regime_transition и фичей. Сохранение для следующих ноутбуков.

## 1. Импорты и настройка путей

In [ ]:
import sys
import os
import numpy as np
import pandas as pd

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

from src.data.dataset_rework_loader import load_dataset_rework, prepare_for_training
from src.features.feature_pipeline import add_features, get_feature_columns
from src.features.warmup_loader import add_warmup_from_bybit, remove_warmup

## 2. Загрузка dataset_rework и подготовка сессий

In [ ]:
data_dir = os.path.join(_root, 'dataset_rework')
df_raw = load_dataset_rework(data_dir=data_dir, verbose=True)
df = prepare_for_training(df=df_raw, verbose=True)
df.head()

## 3. Переименование signal_barrier → rd_regime, rd_regime_transition

In [ ]:
# Если признаки уже есть — не пересчитываем. Иначе строим из signal_barrier/rd_value.
if 'rd_regime' not in df.columns:
    if 'signal_barrier' in df.columns:
        df['rd_regime'] = pd.to_numeric(df['signal_barrier'], errors='coerce').fillna(0)
    else:
        df = df.sort_values(['session_key', 'datetime']).reset_index(drop=True)
        df['rd_regime'] = df.groupby('session_key')['rd_value'].diff().pipe(np.sign).fillna(0)

df['rd_regime'] = np.sign(pd.to_numeric(df['rd_regime'], errors='coerce').fillna(0)).astype(int)

if 'rd_regime_transition' not in df.columns:
    df['rd_regime_prev'] = df.groupby('session_key')['rd_regime'].shift(1)
    df['rd_regime_transition'] = ((df['rd_regime'] != df['rd_regime_prev']) & df['rd_regime_prev'].notna()).astype(int)
    df = df.drop(columns=['rd_regime_prev'], errors='ignore')
else:
    df['rd_regime_transition'] = pd.to_numeric(df['rd_regime_transition'], errors='coerce').fillna(0).astype(int)

print('rd_regime:', df['rd_regime'].value_counts().to_dict())
print('rd_regime_transition (доля переходов):', f"{df['rd_regime_transition'].mean():.4f}")

## 4. Добавление фичей (с warmup Bybit перед расчётом)

In [ ]:
USE_WARMUP = True
WARMUP_SIZE = 60

if USE_WARMUP:
    try:
        # Добавляем 60 минут warmup по каждой сессии, считаем фичи, затем удаляем warmup-строки.
        df_with_warmup = add_warmup_from_bybit(df, warmup_size=WARMUP_SIZE, verbose=True)
        df_feat, enc = add_features(df_with_warmup)
        df = remove_warmup(df_feat)
        print(f'Warmup применён: size={WARMUP_SIZE}. После удаления warmup строк: {len(df):,}')
    except Exception as e:
        print(f'Warmup пропущен ({e}). Считаем фичи без warmup.')
        df, enc = add_features(df)
else:
    df, enc = add_features(df)

feat_cols = get_feature_columns(include_symbol=True)
print(f'Фичей: {len(feat_cols)}')
df[feat_cols[:5] + ['rd_regime', 'rd_regime_transition']].head()

## 5. Сохранение prepared данных

In [ ]:
out_dir = os.path.join(_root, 'outputs')
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'prepared_with_rd_regime.parquet')
df_save = df.copy()
df_save['symbol'] = df_save['symbol'].astype(str)
df_save.to_parquet(out_path, index=False, compression='snappy')
print(f'Сохранено: {out_path} ({len(df):,} строк)')